In [8]:
import pandas as pd
import numpy as np

In [9]:
def normalize(x):
    """Normalize ratings to have 0 mean
    while keeping 0s as 0s"""
    t = x!=0
    mean = x.sum()/t.sum()
    x = np.where(t,x - mean,0)
    return x

In [10]:
class MaximumStructure:
    """Basic implentation of Maximum heap. Allows to iterate over array in
    descending order """
    # INITIAL = 200
    def __init__(self,keys):
        keep=keys>0
        self.inds = np.arange(keys.shape[0])
        self.inds= self.inds[keep]
        self.keys = keys[keep]
        self.shape = self.keys.shape[0]
        self.split = self.shape
        self.heapify()

    def swap(self,i,j):
        self.keys[i],self.keys[j] = self.keys[j],self.keys[i]
        self.inds[j],self.inds[i]=self.inds[i],self.inds[j]

    def sift_down(self,i):
        val_cur = self.keys[i]
        if 2*i+1<self.split:
            left_cur = self.keys[2*i+1]
        else:
            left_cur=-np.inf
        right_cur= self.keys[2*i+2] if 2*i+2<self.split else -np.inf
        if val_cur<right_cur and left_cur<=right_cur:
            selected=2*i+2
        elif val_cur<left_cur:
            selected=2*i+1
        else:
            return
        self.swap(i,selected)
        self.sift_down(selected)

    def heapify(self):
        # i = self.keys.argpartition(MaximumStructure.INITIAL)
        # self.keys=self.keys[i]
        # self.inds=self.inds[i]
        # i=np.argsort(self.keys[-MaximumStructure.INITIAL:])
        # self.keys[- MaximumStructure.INITIAL:]=self.keys[i]
        # self.inds[- MaximumStructure.INITIAL:]=self.inds[i]
        # self.split = self.shape - MaximumStructure.INITIAL
        for j in range((self.split-1)//2,-1,-1):
            self.sift_down(j)


    def extract_max(self):
        self.split-=1
        self.swap(0,self.split)
        self.sift_down(0)
        return self.inds[self.split],self.keys[self.split]

    def extracted(self):
        return self.inds[self.split:],self.keys[self.split:]

    def not_extracted(self):
        return self.split!=0

    def __iter__(self):
        for i in range(self.shape-1,self.split-1,-1):
            yield self.inds[i],self.keys[i]
        while self.split>-self.shape:
            yield self.extract_max()

In [34]:
def recommend(known: list[tuple[str,float]],amount = int) -> list[str]:

    # Get anime
    top_anime = pd.read_csv("top_animes.csv",keep_default_na=False,index_col="title")
    amount_of_anime = top_anime.shape[0]
    top_anime["number"] = np.arange(amount_of_anime)

    #extract user ratings to compare
    cur_known_user_ratings = np.zeros(amount_of_anime)
    titles, ratings = zip(*known)
    titles = np.array(titles)
    cur_known_user_ratings[top_anime.loc[titles,"number"].values] = ratings
    cur_known_user_ratings = normalize(cur_known_user_ratings)

    #extract the training data
    training_ratings = np.load("training_vals.npy")
    #Calculate similarity to training_ratings
    similarities = np.inner(training_ratings,cur_known_user_ratings)
    similarities = MaximumStructure(similarities)

    #predicted ratings
    ratings  = np.repeat(-np.inf,amount_of_anime)

    for ind,row in top_anime.iterrows():

        num = row["number"]

        #No need to predict ratings we know.
        if cur_known_user_ratings[num]!=0:
            continue

        #find rating from the most similar user
        for ind,key in similarities:
            rat = training_ratings[ind,num]
            if rat!=0:
                break
        else:
            continue

        #keep it
        ratings[num] = rat

    # get amount of top rated anime
    indices = np.argpartition(ratings,amount_of_anime-amount)
    indices = indices[-amount:]

    #sort them by ratings in descending order
    ratings = ratings[indices]
    sorted_indices = indices[np.argsort(ratings)]

    #get names
    names = top_anime.index[sorted_indices]
    return list(names)

In [30]:
import kagglehub
import os
svanoo_myanimelist_dataset_path = kagglehub.dataset_download('svanoo/myanimelist-dataset')
top_anime = pd.read_csv("top_animes.csv",keep_default_na=False,index_col="anime_id")
top_anime["number"] = np.arange(top_anime.shape[0])

In [31]:
#TEST, OK To Delete
from tqdm import trange
test_users = pd.read_csv("test_users.csv",index_col="user_id")
test_users = test_users.sample(n=400)

test_values = np.zeros((test_users.shape[0],top_anime.shape[0]),dtype=np.float32)
NUM_USER_ANIME_FILE=70
cols_to_use = ['anime_id', 'user_id', 'score', 'favorite']
RESULT_PATH = "test_user_reviews.csv"
if  os.path.exists(RESULT_PATH):
    os.remove(RESULT_PATH)

for x in trange(NUM_USER_ANIME_FILE):
    file_name = "{}/user_anime{:012d}.csv".format(svanoo_myanimelist_dataset_path,x)
    df = pd.read_csv(
        file_name,
        sep='\t',
        keep_default_na=False,
        na_values=("-------",""),
        low_memory=False,
        usecols=cols_to_use  # Only read needed columns
    ).dropna()
    df = df[df["anime_id"].isin(top_anime.index)]
    df= df[df["user_id"].isin(test_users.index)]
    df.to_csv(RESULT_PATH,mode="a",header = x==0,index=False)


100%|██████████| 70/70 [03:54<00:00,  3.36s/it]


In [45]:
df = pd.read_csv(RESULT_PATH)
from tqdm import tqdm
top_anime = pd.read_csv("top_animes.csv",keep_default_na=False,index_col="anime_id")
top_anime["number"] = np.arange(top_anime.shape[0])
sum_rate = 0
samples = 0
for key,df in tqdm(df.groupby("user_id")):
    df.set_index("anime_id",inplace=True)
    df.score -= df.score.mean()
    df.score/= df.score.std()
    df_to_check=df.sample(frac=0.8)
    l = [(top_anime.loc[ind,"title"],row["score"]) for ind,row in df_to_check.iterrows()]
    recommendations  = recommend(l,30)
    top_anime["anime_id"] = top_anime.index
    top_anime.set_index("title",inplace=True)
    recommendation_id = (top_anime.loc[i,"anime_id"] for i in recommendations)
    known_recommendation_id = [i for i in recommendation_id if i in df.index]
    scores = df.loc[known_recommendation_id]
    sum_rate+=scores.score.sum()
    samples+=scores.shape[0]
    top_anime["title"] = top_anime.index
    top_anime.set_index("anime_id",inplace=True)

100%|██████████| 400/400 [23:50<00:00,  3.58s/it]


In [46]:
print(sum_rate/samples)

0.48037934392805826
